In [ ]:
import os

# Update you config directory as needed
os.environ["PSYNC_CONFIG_DIR"] = "../../../config"

# Collections

The Traktor service implements the standard collection interfaces that are common across all `plistsync` services. By inheriting from the base classes and conforming to the established protocols.

## Locate you traktor library

`plistsync` reads Traktor’s **NML** library/playlist files (usually named `collection.nml`).

Common default locations (may vary by Traktor version and your settings):

- **macOS**
  - `~/Music/Traktor/collection.nml`
  - `~/Documents/Native Instruments/Traktor */collection.nml`
- **Windows**
  - `C:\Users\<you>\Documents\Native Instruments\Traktor *\collection.nml`

If you’re unsure, search your disk for `collection.nml`.

### Option A: Use your live Traktor library file (recommended)

Point `plistsync` at your existing `collection.nml`. Always make a backup before writing.

### Option B: Export an NML from Traktor

If you don’t want to touch your live library, export a playlist/collection from Traktor to an `.nml` file and use that exported file as input.

## Library Collection

The {py:class}`NMLCollection <plistsync.services.traktor.NMLCollection>` represents the Traktor **library** stored in an NML file. It provides:

- Streaming access to all tracks in the Traktor collection (`.tracks`)
- Lookup by local identifiers (currently: file path) via `LocalLookup`
- Access to playlists contained in the NML via `.playlists` / `.get_playlist(...)`

In [ ]:
from plistsync.services.traktor import NMLCollection

library = NMLCollection(path="../../../tests/data/traktor_playlist.nml")

### Track Lookup

The Tidal library collections implements the {py:class}`LocalLookup <plistsync.core.collection.LocalLookup>` protocol, allowing you to search for tracks in a `*.nml` collection. Specifically, we support lookup by `path`.

In [ ]:
from pathlib import PureWindowsPath

# By path (notice we need to use pure windows path here)
path = PureWindowsPath(
    r"D:\SYNC\library\Amoss, Fre4knc\Watermark Volume 2\04 Dragger [1028kbps].flac"
)
if track1 := library.find_by_local_ids({"file_path": path}):
    print(track1)

## Playlist Collection

The {py:class}`NMLPlaylistCollection <plistsync.services.traktor.NMLPlaylistCollection>` represents a playlist in the traktor collection.

### Retrieving playlists

You can retrieve all playlists saved in a collection using the library's {py:meth}`NMLCollection.playlists <plistsync.services.traktor.NMLCollection.playlists>` property.


In [ ]:
playlists = library.playlists
for playlist in playlists:
    print(f"Playlist: {playlist.name} ({len(playlist)} tracks)")

If you just want a specific playlist, you can use the library's {py:meth}`NMLCollection.get_playlist <plistsync.services.traktor.NMLCollection.get_playlist>` method to retrieve a playlist.

In [ ]:
if pl := library.get_playlist(
    name="Silvester Full Playthrough"
    # uuid = "6868ecd66b354d37a33b965dae7a82e7"
):
    print(f"Found playlist: {pl.name} ({len(pl)} tracks)")
else:
    print("Playlist not found.")

:::{note}
This method supports lookup by ``name=`` or ``uuid=``.
:::

### Creating playlists

Currently we do not support creating new playlists with a unified interface. We will reevaluate this at a later point in time. For now you need to create a playlist by creating a playlist object and attaching it to the libary.

In [ ]:
from plistsync.services.traktor import NMLPlaylistCollection

# Create new playlist
pl = NMLPlaylistCollection(library, name="New playlist")
# Upsert in nml collection (will overwrite base)
library.upsert_playlist(pl)

To persist changes to the collection.nml on disk call {py:meth}`NMLCollection.write <plistsync.services.traktor.NMLCollection.write>` method.

In [ ]:
library.write()

### Updating a playlist

For updating a playlist, you should use the playlist's {py:meth}`NMLPlaylistCollection.edit <plistsync.services.tidal.NMLPlaylistCollection.edit>` context manager. This ensures that all changes are properly saved back to Tidal when you exit the context. This also minifies changes and therefore reduces API calls.


In [ ]:
# Updating a playlist description/name
with pl.edit():
    pl.name = "Created via plistsync"

pl.name

You can add tracks to a playlist using the same context manager, again this will add the tracks when you exit the context.

In [ ]:
from plistsync.services.traktor import NMLPlaylistTrack

# Add a track to the playlist
new_track = library.find_by_local_ids({"file_path": path})
assert new_track is not None

with pl.edit():
    # TODO: We should reevaluate the type hierarchy here.
    # for now, we have to manually cast PlaylistTrack to NMLPlaylistTrack.
    # this will become more convenient, eventually.
    pl.tracks.append(NMLPlaylistTrack.from_track(new_track))
    pl.tracks.append(NMLPlaylistTrack.from_track(new_track))

To reorder tracks in a playlist, you can change the order of the {py:attr}`NMLPlaylistCollection.tracks <plistsync.services.traktor.NMLPlaylistCollection.tracks>` list within the context manager.

In [ ]:
with pl.edit():
    pl.tracks.insert(0, pl.tracks.pop())

:::{note}
To persist changes to disk, make sure to call the `library.write()` method on the associated NMLCollection.
:::

In [ ]:
library.write()